# xenquaco

This notebook is a comprehensive guide for using **xenquaco**, a QC pipeline for 10x Xenium spatial transcriptomics experiments.

Current functionality includes:
- **Transcript density**: on-tissue transcript density per µm², normalized by gene count
- **Pixel classification**: classifies every pixel of your section as tissue, damage, detachment, or ventricles

Most functionality requires only the `transcripts.parquet` file from a Xenium output directory. The pixel classification workflow additionally requires:
- The `morphology_focus.ome.tif` DAPI image (Xenium output)
- [ilastik](https://www.ilastik.org/download) installed, with the path to its executable provided

## Installation

Clone the repo and install in editable mode:

```bash
git clone https://github.com/polsen99/xenquaco.git
cd xenquaco
pip install -e .
```

## Import xenquaco

In [ ]:
import xenquaco as xqc
from pathlib import Path
import matplotlib.pyplot as plt
import tifffile as tiff
import numpy as np
import os

## Creating a XeniumExperiment object

The `XeniumExperiment` object is the main entry point for xenquaco. It is initialized with the path to `transcripts.parquet` (or a pre-loaded DataFrame).

On initialization, it will:
- Read the transcripts table
- Filter transcripts with QV < 20
- Remove control codewords (NegControlCodeword, NegControlProbe, UnassignedCodeword, DeprecatedCodeword)
- Build the FOVs dataframe
- (Optional) Generate the transcript tissue mask via ilastik

In [ ]:
# Path to Xenium output directory
transcripts_path = 'path/to/your_xenium_output/transcripts.parquet'

# (Optional) Path to ilastik executable — required for pixel classification
ilastik_program_path = 'path/to/ilastik'

# (Optional) Path to DAPI image — required for pixel classification
dapi_high_res_image_path = 'path/to/your_xenium_output/morphology_focus.ome.tif'

# Output directory for xenquaco results
output_dir = Path(os.getcwd(), 'qc_output')

In [ ]:
# Initialize with path to transcripts.parquet
exp = xqc.experiment.XeniumExperiment(
    transcripts_input=transcripts_path,
    ilastik_program_path=ilastik_program_path,
    dapi_high_res_image_path=dapi_high_res_image_path,
    output_dir=output_dir
)

Let's look at some basic experiment attributes:

In [ ]:
print(f'Total transcripts detected:    {exp.total_transcript_count:,}')
print(f'Filtered transcripts (QV≥20, no controls): {exp.filtered_transcript_count:,}')
print(f'Number of genes:               {exp.n_genes}')

The `exp.filtered_transcripts` DataFrame uses Xenium-native column names: `feature_name`, `x_location`, `y_location`, `z_location`, `qv`, `fov_name`.

In [ ]:
exp.filtered_transcripts.head()

The `exp.fovs_df` DataFrame contains coordinates, transcript counts, and neighbor information per FOV:

In [ ]:
exp.fovs_df.head()

## Running all QC

`exp.run_all_qc()` is the most convenient method — it runs the full pipeline and saves outputs to `output_dir`:
1. **Pixel classification**: generates tissue, damage, detachment, and ventricle masks
2. **On-tissue transcript density**: transcripts per µm², normalized by gene count
3. **Figures**: saves pixel classification plot and transcript overview
4. **QC summary**: saves `qc_summary.json` with all metrics

In [ ]:
exp.run_all_qc(
    run_pixel_classification=True,
    plot_figures=True,
    save_metrics=True
)

---
# Individual Metrics

With an initialized `XeniumExperiment`, we can also compute metrics individually.

## Transcripts Overview

In [ ]:
xqc.figures.transcripts_overview(exp.filtered_transcripts, title='Filtered Transcripts')
plt.show()

## On-Tissue Transcript Density

The binary tissue mask generated via ilastik approximates the tissue boundary. We use this to compute on-tissue transcript density.

In [ ]:
ts_image = tiff.imread(exp.transcripts_image_path)
ts_mask = tiff.imread(exp.transcripts_mask_path)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(ts_image.T, cmap='bone_r')
ax[0].axis('off')
ax[0].set_title('Transcript Locations')
ax[1].imshow(ts_mask.T, cmap='bone_r')
ax[1].axis('off')
ax[1].set_title('Transcripts Binary Mask')
plt.show()

In [ ]:
transcript_density = xqc.experiment.get_transcript_density(ts_image, ts_mask)
transcript_density_per_gene = transcript_density / exp.n_genes

print(f'On-tissue transcript density:          {np.round(transcript_density, 4)} transcripts/µm²')
print(f'On-tissue density per gene:            {np.round(transcript_density_per_gene, 6)} transcripts/µm²/gene')

## Pixel Classification

The pixel classification workflow classifies every pixel of the section into one of five categories:

| Value | Label |
|-------|-------|
| 0 | Off-tissue |
| 1 | Damage |
| 2 | Tissue |
| 3 | Detachment |
| 4 | Ventricles |

**Note on detachment in Xenium:** Unlike MERSCOPE, Xenium experiments are not prepared with a gel. However, tissue can still detach from the slide during imaging, producing regions with DAPI signal but no transcripts. This mask captures those regions.

**Note on ilastik models:** The bundled models were trained on MERSCOPE data and may need retraining on Xenium data for optimal performance.

In [ ]:
# Run pixel classification on its own (already run if run_all_qc was called above)
exp.run_full_pixel_classification(save_metrics=True)

In [ ]:
xqc.figures.plot_full_pixel_fig(
    exp.pixel_classification,
    exp.dapi_mask,
    exp.transcripts_mask,
    exp.detachment_mask,
    exp.transcripts_percent,
    exp.detachment_percent,
    exp.damage_mask,
    exp.ventricle_mask,
    exp.damage_percent,
    exp.ventricle_percent
)

We can also look at the pixel class areas and percentages of ideal tissue area directly:

In [ ]:
print(f'Tissue area:      {exp.transcripts_area:,.0f} µm²  ({exp.transcripts_percent}%)')
print(f'Damage area:      {exp.damage_area:,.0f} µm²  ({exp.damage_percent}%)')
print(f'Detachment area:  {exp.detachment_area:,.0f} µm²  ({exp.detachment_percent}%)')
print(f'Ventricle area:   {exp.ventricle_area:,.0f} µm²  ({exp.ventricle_percent}%)')
print(f'Total area:       {exp.total_area:,.0f} µm²')

## Ventricle Genes

The ventricle mask is generated using a list of known ventricle marker genes. By default, xenquaco uses a set of mouse ventricle markers — you can override this with your own list when initializing the experiment:

In [ ]:
# Example with a custom ventricle gene list
# exp = xqc.experiment.XeniumExperiment(
#     transcripts_input=transcripts_path,
#     ilastik_program_path=ilastik_program_path,
#     dapi_high_res_image_path=dapi_high_res_image_path,
#     output_dir=output_dir,
#     ventricle_genes_list=['Crb2', 'Glis3', 'Hdc']  # your custom list
# )

## QC Summary

All metrics are saved to `qc_summary.json` in the output directory when `save_metrics=True`:

In [ ]:
import json

with open(Path(output_dir, 'qc_summary.json'), 'r') as f:
    qc_summary = json.load(f)

for key, val in qc_summary.items():
    print(f'{key}: {val}')